<a href="https://colab.research.google.com/github/JamesMartinOU/PublicRedditSentimentAnalysis/blob/main/RedditCreateRollingAverageTable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Python libraries
!pip install mysql-connector-python
!pip install sqlalchemy pymysql

In [ ]:
# Import Python libraries
import mysql.connector
import pandas as pd
from google.colab import files
from sqlalchemy import create_engine
import pymysql

In [ ]:
# RDS MySQL connection details

In [ ]:
# Connect to the MySQL database
conn = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

# Query to create sentiment fact table
query_create_fact_table_weekly_rolling_avg = """
CREATE TABLE reddit_sentiment_fact_table_weekly_rolling_avg (
  company_name VARCHAR(255),
  year INT,
  week TINYINT,
  total_score INT,
  total_count INT,
  weekly_sentiment DECIMAL(5,3),
  rolling_avg_sentiment DECIMAL(5,3)
);
"""



# Step One query
query_rolling_avg_select = """
WITH RECURSIVE calendar AS (
  -- start with the earliest year/week
  SELECT
    MIN(year) AS year,
    MIN(week) AS week
  FROM reddit_sentiment_fact_table_weekly

  UNION ALL

  -- increment week, roll over year when needed
  SELECT
    CASE WHEN week = 52 THEN year + 1 ELSE year END,
    CASE WHEN week = 52 THEN 1 ELSE week + 1 END
  FROM calendar
  WHERE (year < (SELECT MAX(year) FROM reddit_sentiment_fact_table_weekly))
     OR (year = (SELECT MAX(year) FROM reddit_sentiment_fact_table_weekly)
         AND week < (SELECT MAX(week) FROM reddit_sentiment_fact_table_weekly))
),

company_weeks AS (
  SELECT DISTINCT company_name FROM reddit_sentiment_fact_table_weekly
),
all_weeks AS (
  SELECT
    cw.company_name,
    c.year,
    c.week
  FROM calendar c
  CROSS JOIN company_weeks cw
),

weekly_sentiment AS (
  SELECT
    company_name,
    year,
    week,
    SUM(
      CASE
        WHEN sentiment = 'Positive' THEN 1
        WHEN sentiment = 'Negative' THEN -1
        ELSE 0
      END * count
    ) AS total_score,
    SUM(count) AS total_count
  FROM reddit_sentiment_fact_table_weekly
  GROUP BY company_name, year, week
),

joined_timeline AS (
  SELECT
    aw.company_name,
    aw.year,
    aw.week,
    ws.total_score,
    ws.total_count
  FROM all_weeks aw
  LEFT JOIN weekly_sentiment ws
    ON aw.company_name = ws.company_name
   AND aw.year = ws.year
   AND aw.week = ws.week
)

SELECT
  company_name,
  year,
  week,
  total_score,
  total_count,
  ROUND(total_score / NULLIF(total_count, 0), 3) AS weekly_sentiment,

  -- 3-week rolling average
  ROUND(
    AVG(total_score * 1.0 / NULLIF(total_count, 0)) OVER (
      PARTITION BY company_name
      ORDER BY year, week
      ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ),
    3
  ) AS rolling_avg_sentiment

FROM joined_timeline
ORDER BY company_name, year, week;

"""


# Insert into target table
insert_query = """
INSERT INTO reddit_sentiment_fact_table_weekly_rolling_avg
(company_name, year, week, total_score, total_count, weekly_sentiment, rolling_avg_sentiment)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""


# Create cursor and execute
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS reddit_sentiment_fact_table_weekly_rolling_avg;")
print("Table dropped successfully")

# Execute rolling avg query and fetch results
cursor.execute(query_rolling_avg_select)
rows = cursor.fetchall()

# Create DataFrame from query results
df_sentiment_fact = pd.DataFrame(rows, columns=[
    'company_name', 'year', 'week',
    'total_score', 'total_count',
    'weekly_sentiment', 'rolling_avg_sentiment'
])

# Forward-fill rolling average before writing to DB
df_sentiment_fact['rolling_avg_sentiment_filled'] = (
    df_sentiment_fact
    .groupby('company_name')['rolling_avg_sentiment']
    .transform(lambda x: x.fillna(method='ffill'))
)


# Write to MySQL table
engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}')
df_sentiment_fact.to_sql('reddit_sentiment_fact_table_weekly_rolling_avg', con=engine, if_exists='replace', index=True)
print("Table updated successfully")

# Close old connection and cursor
cursor.close()
conn.close()

# Connect to the MySQL database
conn = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)

# Execute the SELECT query
cursor = conn.cursor()
cursor.execute("SELECT * FROM reddit_sentiment_fact_table_weekly_rolling_avg;")

# Fetch all rows and get column names
rows = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]

# Create DataFrame
df_extracted = pd.DataFrame(rows, columns=columns)

# Cleanup
cursor.close()
conn.close()

# Preview the data
print(df_extracted.head())

Table dropped successfully


<ipython-input-5-f5ced4b7d287>:140: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .transform(lambda x: x.fillna(method='ffill'))


Table updated successfully
   index company_name  year  week total_score total_count weekly_sentiment  \
0      0       AbbVie  2008     0        None        None             None   
1      1       AbbVie  2008     1        None        None             None   
2      2       AbbVie  2008     2        None        None             None   
3      3       AbbVie  2008     3        None        None             None   
4      4       AbbVie  2008     4        None        None             None   

  rolling_avg_sentiment rolling_avg_sentiment_filled  
0                  None                         None  
1                  None                         None  
2                  None                         None  
3                  None                         None  
4                  None                         None  
